# Huấn luyện NS-MGAT trên Google Colab

> Dùng cho **mọi** thí nghiệm cần GPU: `phobert` (CD1.5), `asgcn` + `asgcn_linked` (CD1.6a),
> `senticgcn` (CD1.6b). Chỉ đổi hai chữ ở ô "Chạy một seed".

**Vì sao không chạy ở máy:** đo thật trên CPU 8 nhân của máy học viên — 31,8 giây/bước,
tức **52,8 giờ cho một seed**, 6,6 ngày cho 3 seed. Colab T4 mất ~30–45 phút/seed.

## Chuẩn bị một lần (làm ở máy, trước khi mở notebook này)

1. **Đẩy code lên GitHub** — Colab kéo code từ đó:
   ```powershell
   git push
   ```
2. **Tải dữ liệu lên Drive.** `data/processed/*.jsonl` không nằm trong git (đã `.gitignore`),
   nên phải đưa lên Drive tay, **một lần duy nhất**:
   - Tạo trên Google Drive thư mục `nsmgat/data/processed/`
   - Tải lên 3 file trong `data/processed/` ở máy: `visfd_train.jsonl`,
     `visfd_dev.jsonl`, `visfd_test.jsonl` (tổng 45 MB)

## Cách notebook này giữ kết quả khi Colab ngắt phiên

Colab xoá sạch máy ảo khi hết phiên. Nên `results/`, `checkpoints/`, `logs/` được **nối
sang Drive bằng symlink** — ghi vào chúng là ghi thẳng vào Drive.

Nối thư mục chứ **không** sửa đường dẫn trong config, vì `results_dir` và `ckpt_dir` nằm
trong `config_hash`: sửa là lần chạy Colab mang vân tay khác với `configs/phobert.yaml`,
và không còn đối chiếu được với các lần chạy khác.

## 1. Kiểm GPU — làm trước mọi thứ khác

In [ ]:
# Chay CPU thi mot seed mat ~53 gio. Dung lai ngay con hon chay 3 tieng roi moi biet.
import torch, subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "(khong thay nvidia-smi)")
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "KHONG CO GPU. Vao Runtime > Change runtime type > Hardware accelerator = GPU (T4), "
    "roi chay lai o nay."
)
print("OK — co GPU, chay tiep duoc.")

## 2. Gắn Drive và dựng cây thư mục

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

GOC_DRIVE = Path("/content/drive/MyDrive/nsmgat")
for ten in ("data/processed", "results", "checkpoints", "logs"):
    (GOC_DRIVE / ten).mkdir(parents=True, exist_ok=True)

co = sorted(p.name for p in (GOC_DRIVE / "data/processed").glob("*.jsonl"))
print("Thu muc Drive:", GOC_DRIVE)
print("Du lieu da co :", co or "(CHUA CO — xem phan 'Chuan bi mot lan' o dau notebook)")

## 3. Lấy code

Cách chính: kéo từ GitHub. Nếu kho riêng tư hoặc bạn chưa `git push`, dùng ô **3b**.

In [ ]:
import subprocess
from pathlib import Path

REPO = Path("/content/nsmgat")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/2611326-NguyenMinhTrong/nsmgat-vietnamese-absa.git", str(REPO)], check=True)

# In ra commit dang chay — de sau nay truy duoc ket qua nay sinh tu ma nguon nao
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

### 3b. *(chỉ khi ô trên không dùng được)* Lấy code từ Drive

Ở máy: nén thư mục repo (bỏ `.venv`, `data`, `results`, `checkpoints`) thành `nsmgat.zip`,
tải lên `Drive/nsmgat/nsmgat.zip`, rồi chạy ô này.

In [ ]:
# !unzip -q -o /content/drive/MyDrive/nsmgat/nsmgat.zip -d /content/nsmgat

## 4. Cài thư viện

Chỉ cài thứ Colab **chưa có**. Không chạy `pip install -r requirements.txt`: file đó có
`torch`, cài đè lên bản torch-CUDA sẵn của Colab là hỏng GPU.

Dòng `pip install -e` là **bắt buộc**, không phải tuỳ chọn: mã nguồn nằm ở `src/nsmgat/`
(src-layout), không cài thì `python -m nsmgat.train` báo `ModuleNotFoundError: nsmgat`.

In [ ]:
%pip install -q transformers pyyaml
%pip install -q -e /content/nsmgat   # BAT BUOC — src-layout, khong cai thi khong import duoc

import importlib, nsmgat
importlib.reload(nsmgat)
print("nsmgat import duoc tu:", nsmgat.__file__)

## 5. Nối thư mục sang Drive

Sau ô này, mọi thứ mô hình ghi ra (`results/`, `checkpoints/`, `logs/`) đi thẳng vào Drive —
Colab ngắt phiên cũng không mất.

In [ ]:
import os
from pathlib import Path

REPO = Path("/content/nsmgat")
GOC_DRIVE = Path("/content/drive/MyDrive/nsmgat")
os.chdir(REPO)

for ten in ("data/processed", "results", "checkpoints", "logs"):
    dich = REPO / ten
    dich.parent.mkdir(parents=True, exist_ok=True)
    if dich.is_symlink():
        dich.unlink()
    elif dich.exists():
        # Thu muc that (vd data/processed rong tu git) -> doi ten de khoi mat
        dich.rename(dich.with_name(dich.name + "_local"))
    os.symlink(GOC_DRIVE / ten, dich)
    print(f"{ten:18} -> {GOC_DRIVE / ten}")

# Kiem du lieu that su doc duoc, khong chi la co file
for ten in ("train", "dev", "test"):
    p = REPO / "data/processed" / f"visfd_{ten}.jsonl"
    assert p.exists(), f"THIEU {p} — tai len Drive truoc (xem dau notebook)"
    print(f"  visfd_{ten}: {sum(1 for _ in p.open(encoding='utf-8')):,} dong")

## 6. Chạy một seed

Đổi `CONFIG` / `MODEL` khi dùng cho thí nghiệm khác. Mỗi seed ~30–45 phút trên T4.

**Đừng đóng tab.** Colab ngắt phiên sau ~90 phút không tương tác; một seed vừa đủ một
phiên. Bị ngắt giữa chừng thì chạy ô **7** (chạy tiếp), không mất gì.

In [ ]:
CONFIG = "configs/phobert.yaml"
MODEL = "phobert"
SEED = 42

!python -m nsmgat.train --config {CONFIG} --model {MODEL} --seed {SEED}

## 7. Chạy tiếp sau khi bị ngắt

`--resume` đọc `checkpoints/<exp>/seed<N>/last.pt` trên Drive và chạy tiếp **đúng chỗ đang
dở**, gồm cả trạng thái sinh số ngẫu nhiên — kết quả giống hệt chạy liền mạch
(`tests/test_resume.py` khoá điều này).

Đổi config rồi thì nó **từ chối** chạy tiếp: ghép nửa đầu cấu hình này với nửa sau cấu
hình khác thì con số thu được vô nghĩa.

In [ ]:
!python -m nsmgat.train --config {CONFIG} --model {MODEL} --seed {SEED} --resume

## 8. Hai seed còn lại

In [ ]:
for seed in (1337, 2024):
    print(f"\n{'='*70}\n  SEED {seed}\n{'='*70}")
    !python -m nsmgat.train --config {CONFIG} --model {MODEL} --seed {seed}

## 9. Xem kết quả 3 seed

In [ ]:
import json, statistics
from pathlib import Path

EXP = MODEL
hang = []
for seed in (42, 1337, 2024):
    p = Path("results") / EXP / f"seed{seed}" / "metrics.json"
    if not p.exists():
        print(f"seed {seed}: chua co ket qua")
        continue
    m = json.loads(p.read_text(encoding="utf-8"))
    hang.append((seed, m["test"]["accuracy"], m["test"]["macro_f1"],
                 m["test"]["f1_per_class"], m.get("config_hash", "?")[:12],
                 m.get("train_time_sec", 0)))

print(f"{'seed':>6} {'acc':>8} {'macro-F1':>9} {'F1(NEU)':>8} {'van tay':>14} {'phut':>7}")
for seed, acc, f1, per_class, h, t in hang:
    print(f"{seed:>6} {acc:>8.4f} {f1:>9.4f} {per_class[1]:>8.3f} {h:>14} {t/60:>7.1f}")

if len(hang) == 3:
    for ten, i in (("accuracy", 1), ("macro-F1", 2)):
        gia_tri = [h[i] for h in hang]
        print(f"\n{ten}: {statistics.mean(gia_tri):.4f} +/- {statistics.stdev(gia_tri):.4f}")
    print("\nSo sanh: lexicon 0,7469 | bilstm 0,8642 (accuracy, trung binh 3 seed)")
    print("Tran mu khia canh tren test: 0,8088 — bilstm da vuot.")

## 10. Mang kết quả về máy

`results/` đã nằm trên Drive rồi. Ở **máy**, chép vào repo cho đúng chỗ:

```
Drive/nsmgat/results/phobert/seed42/{metrics.json, predictions.jsonl}
   →  <repo>/results/phobert/seed42/
```

Chỉ cần **`metrics.json`** và **`predictions.jsonl`**. **Đừng chép `checkpoints/`** vào repo:
`best.pt` của phobert nặng ~540 MB, và `.gitignore` đã chặn `checkpoints/`.

Về máy rồi thì kiểm lại bằng:

```powershell
.\.venv\Scripts\python.exe -m pytest -q
```

## 11. Dọn Drive khi chạy xong cả 3 seed

`last.pt` của phobert nặng **~1,6 GB mỗi seed** (trọng số + 2 trạng thái Adam). Ba seed là
gần 5 GB — Drive miễn phí có 15 GB.

Xong hẳn rồi thì xoá `last.pt`; **giữ `best.pt`** nếu còn muốn chạy lại đánh giá
(`--no-train`). Ô dưới chỉ *liệt kê* — muốn xoá thì bỏ dấu `#` ở dòng cuối.

In [ ]:
from pathlib import Path

for p in sorted(Path("/content/drive/MyDrive/nsmgat/checkpoints").rglob("*.pt")):
    print(f"{p.stat().st_size/1e9:6.2f} GB  {p}")

# for p in Path("/content/drive/MyDrive/nsmgat/checkpoints").rglob("last.pt"): p.unlink()